In [1]:
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go

# set directories
WD_junxi = Path('PATH_TO_DATA')
WD = WD_junxi
data_dir = Path(WD/'EntTemplates/Analysis/python_Patent/data/')
output_dir = Path(WD/'EntTemplates/Analysis/python_Patent/output/')


In [2]:
#df_predictions = pd.read_csv(WD_junxi / 'EntTemplates/Analysis/python_BERT/data_patent/positive_results_100k_.csv', dtype=str)
df_predictions = pd.read_stata(WD_junxi / 'EntTemplates/Analysis/python_BERT/data_patent/positive_results_500k_new.dta')
# drop "index" column if exists
if 'index' in df_predictions.columns:
    df_predictions = df_predictions.drop(columns=['index'])
df_patent = pd.read_csv(data_dir / 'g_patent.tsv', sep='\t', dtype=str)
# merge to get abstract
df_predictions = df_predictions.merge(df_patent[['patent_id','patent_abstract']], on='patent_id', how='left')

In [4]:
# get the first four digit of patent_date, and only keep if the first four digit is between 2000 and 2013. Drop if missing values
df_predictions = df_predictions[df_predictions['patent_year']==2010]
df_predictions = df_predictions[['patent_id', 'patent_date', 'patent_year','disambig_country','file',  'patent_abstract']]
# rename file to sector
df_predictions = df_predictions.rename(columns={'file': 'sector'})

del df_patent

## Within global pairs

In [ ]:
from itertools import combinations
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

model = SentenceTransformer("all-distilroberta-v1")

df_sbert_input = (
    df_predictions
    .dropna(subset=["patent_abstract", "disambig_country", "sector"])
    .reset_index(drop=True)
 )

groups = list(df_sbert_input.groupby("sector"))
output_dir_progress = output_dir / "progress_500k_2010"
output_dir_progress.mkdir(parents=True, exist_ok=True)

for sector, grp in tqdm(groups, desc="Sectors"):
    progress_file = output_dir_progress / f"pairwise_{sector}.parquet"
    if progress_file.exists():
        tqdm.write(f"Skip {sector}: {progress_file.name} exists")
        continue

    grp = grp.reset_index(drop=True)
    embeddings = model.encode(
        grp["patent_abstract"].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    total_pairs = len(grp) * (len(grp) - 1) // 2

    sector_pairs = []
    for (i, emb_i), (j, emb_j) in tqdm(
        combinations(enumerate(embeddings), 2),
        total=total_pairs,
        desc=f"{sector} pairs",
        leave=False,
    ):
        if grp.loc[i, "disambig_country"] == grp.loc[j, "disambig_country"]:
            continue
        score = float(np.dot(emb_i, emb_j))
        sector_pairs.append({
            "patent_id1": grp.loc[i, "patent_id"],
            "patent_id2": grp.loc[j, "patent_id"],
            "country1": grp.loc[i, "disambig_country"],
            "country2": grp.loc[j, "disambig_country"],
            "sbert_score": score,
            "sector": sector,
        })

    df_sector_pairs = pd.DataFrame(sector_pairs)
    df_sector_pairs.to_parquet(progress_file, index=False)
    tqdm.write(f"Wrote {len(df_sector_pairs)} rows for {sector} -> {progress_file.name}")

    del sector_pairs
    del df_sector_pairs
    del embeddings

progress_files = sorted(output_dir_progress.glob("pairwise_*.parquet"))
if not progress_files:
    raise FileNotFoundError(f"No progress files found in {output_dir_progress}")

df_pairwise_similarity = pd.concat(
    [pd.read_parquet(pf) for pf in progress_files],
    ignore_index=True,
)

print(f"Total pairs across sectors: {len(df_pairwise_similarity)}")

df_pairwise_similarity.to_parquet(
    output_dir / "patent_pairwise_similarity_2010_500k_new_china.parquet",
    index=False,)


In [8]:
df_pairwise_similarity = pd.read_parquet(
    output_dir / 'patent_pairwise_similarity_2010_500k_new_china.parquet')

In [4]:
df_pairwise_similarity = pd.read_parquet(
    output_dir / 'patent_pairwise_similarity_2000_2013_100k_new_china.parquet')

In [15]:
# Normalize country ordering (alphabetical) before aggregation
df_pairs = df_pairwise_similarity.copy()
swap_mask = df_pairs["country1"] > df_pairs["country2"]

# swap country and patent columns where needed
df_pairs.loc[swap_mask, ["country1", "country2"]] = df_pairs.loc[swap_mask, ["country2", "country1"]].to_numpy()
df_pairs.loc[swap_mask, ["patent_id1", "patent_id2"]] = df_pairs.loc[swap_mask, ["patent_id2", "patent_id1"]].to_numpy()

agg_country_sector = (
    df_pairs
    .groupby(["country1", "country2", "sector"])["sbert_score"]
    .agg(
        mean="mean",
        median="median",
        top25=lambda x: x.quantile(0.75),
        top10=lambda x: x.quantile(0.90),
        top5=lambda x: x.quantile(0.95),
        top1=lambda x: x.quantile(0.99),
    )
    .reset_index()
)

agg_country_sector.head()

,country1,country2,sector,mean,median,top25,top10,top5,top1
0,AE,AR,EnterpriseHealthCustomerAcquisitionToolsSchedu...,0.126767,0.126767,0.126767,0.126767,0.126767,0.126767
1,AE,AR,IoTIoTHardwareSensorsSensorSystems,-0.027263,-0.027263,-0.027263,-0.027263,-0.027263,-0.027263
2,AE,AR,RetailHealthTechDietarySupplementsVitaminsSupp...,0.220827,0.220827,0.220827,0.220827,0.220827,0.220827
3,AE,AT,AIMLHorizontalPlatformsComputerVision,0.233110,0.252777,0.260098,0.271995,0.275960,0.279133
4,AE,AT,AgTechAgrifinanceeCommerceAgribusinessmarketpl...,0.108375,0.108375,0.144418,0.166044,0.173253,0.179020


In [16]:
agg_country_sector.to_stata(
    output_dir / 'patent_countrysector_similarity_2010_500k_new_china.dta',
    write_index=False,)

## Cross-set similarity: CN vs global (sector-by-sector), pre-2013

This section mirrors the style of **Within global pairs**:
- group by `sector`
- encode abstracts within each sector
- compute cosine similarity via dot-product of normalized SBERT embeddings

It produces **all CN↔global pairs within the same sector** (not top-k). Output is written as one parquet file per sector to avoid holding an enormous list of pairs in RAM.

In [ ]:
import numpy as np
from tqdm.auto import tqdm

# Load CN positives produced by patent_Chinese_prediction.ipynb
cn_path = output_dir / "positive_results_China_50k.csv"
df_cn = pd.read_csv(cn_path, dtype=str)

# Expected columns: appid, abstract, sector
df_cn = df_cn.rename(columns={"appid": "patent_id", "abstract": "patent_abstract"})
for col in ["patent_id", "patent_abstract", "sector"]:
    if col not in df_cn.columns:
        raise ValueError(f"CN file missing expected column: {col}. Columns={list(df_cn.columns)}")

df_cn["patent_id"] = df_cn["patent_id"].astype(str)
df_cn["patent_abstract"] = df_cn["patent_abstract"].fillna("").astype(str)
df_cn = df_cn[df_cn["patent_abstract"].str.strip() != ""].copy()
df_cn["disambig_country"] = "CN"

# Attach patent year from g_patent.tsv (usecols keeps memory down)
g_patent_dates = pd.read_csv(
    data_dir / "g_patent.tsv",
    sep="\t",
    usecols=["patent_id", "patent_date"],
    dtype=str,
    low_memory=False,
)
g_patent_dates = g_patent_dates.rename(columns={"patent_id": "patent_id"})
g_patent_dates["patent_id"] = g_patent_dates["patent_id"].astype(str)
g_patent_dates["patent_year"] = g_patent_dates["patent_date"].str[:4]

df_cn = df_cn.merge(g_patent_dates[["patent_id", "patent_year"]], on="patent_id", how="left")
df_cn = df_cn.dropna(subset=["patent_year"])
df_cn = df_cn[df_cn["patent_year"].between("2000", "2013")].reset_index(drop=True)

print("CN (pre-2013) patents:", df_cn.shape)
display(df_cn[["patent_id", "patent_year", "sector"]].head())
del g_patent_dates

CN (pre-2013) patents: (32041, 6)


,patent_id,patent_year,sector
0,8177002,2012,MobilityTechIndustrialEquipmentIndustrialEquip...
1,8370994,2013,MobilityTechIndustrialEquipmentIndustrialEquip...
2,7850236,2010,MobilityTechIndustrialEquipmentIndustrialEquip...
3,8376621,2013,MobilityTechIndustrialEquipmentIndustrialEquip...
4,8205298,2012,MobilityTechIndustrialEquipmentIndustrialEquip...


In [ ]:
# Prepare global pre-2013 set (100k) from df_predictions already loaded above
df_global = df_predictions.copy()

# Ensure consistent schema
if "sector" not in df_global.columns and "file" in df_global.columns:
    df_global = df_global.rename(columns={"file": "sector"})

df_global["patent_id"] = df_global["patent_id"].astype(str)
df_global["patent_abstract"] = df_global["patent_abstract"].fillna("").astype(str)
df_global["patent_year"] = df_global["patent_date"].astype(str).str[:4]

df_global = df_global.dropna(subset=["patent_year"])
df_global = df_global[df_global["patent_year"].between("2000", "2013")].copy()
df_global = df_global.dropna(subset=["patent_abstract", "disambig_country", "sector"]).reset_index(drop=True)
df_global = df_global[df_global["patent_abstract"].str.strip() != ""]

print("Global (pre-2013) patents:", df_global.shape)
display(df_global[["patent_id", "patent_year", "disambig_country", "sector"]].head())

Global (pre-2013) patents: (256176, 9)


,patent_id,patent_year,disambig_country,sector
0,6108128,2000,JP,FintechWealthtechDigitalAdvisory
1,7278192,2007,DE,FintechWealthtechDigitalAdvisory
2,6754246,2004,JP,FintechWealthtechDigitalAdvisory
3,8463086,2013,JP,FintechWealthtechDigitalAdvisory
4,8562219,2013,JP,FintechWealthtechDigitalAdvisory


In [ ]:
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np

# Compute CN ↔ global similarities within the same sector, mirroring the within-global logic.
# Writes one parquet per sector to avoid huge in-memory objects.
 
model = SentenceTransformer("all-distilroberta-v1")

# Ensure minimal required columns are present
need_cn = ["patent_id", "patent_abstract", "sector", "patent_year"]
need_gl = ["patent_id", "patent_abstract", "sector", "disambig_country", "patent_year"]
for c in need_cn:
    if c not in df_cn.columns:
        raise ValueError(f"df_cn missing column: {c}")
for c in need_gl:
    if c not in df_global.columns:
        raise ValueError(f"df_global missing column: {c}")

# Group by sector
cn_groups = dict(tuple(df_cn.groupby("sector")))
gl_groups = dict(tuple(df_global.groupby("sector")))
common_sectors = sorted(set(cn_groups.keys()).intersection(gl_groups.keys()))
print("Common sectors:", len(common_sectors))

out_dir = output_dir / "cn_vs_global_pairwise_similarity_2000_2013_by_sector_new"
out_dir.mkdir(parents=True, exist_ok=True)

sector_stats = []
for sector in tqdm(common_sectors, desc="Sectors"):
    cn_grp = cn_groups[sector].reset_index(drop=True)
    gl_grp = gl_groups[sector].reset_index(drop=True)

    # Encode both sides (normalized => cosine = inner product)
    emb_cn = model.encode(
        cn_grp["patent_abstract"].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    emb_gl = model.encode(
        gl_grp["patent_abstract"].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    # For each CN patent, compute similarity to all global patents in the same sector
    rows = []
    gl_ids = gl_grp["patent_id"].astype(str).to_numpy()
    gl_country = gl_grp["disambig_country"].astype(str).to_numpy()
    cn_ids = cn_grp["patent_id"].astype(str).to_numpy()

    for i, emb_i in enumerate(emb_cn):
        scores = np.dot(emb_gl, emb_i)  # (n_gl,)
        for j, score in enumerate(scores):
            rows.append({
                "cn_patent_id": cn_ids[i],
                "patent_id": gl_ids[j],
                "disambig_country": gl_country[j],
                "sector": sector,
                "sbert_score": float(score),
            })

    df_sector = pd.DataFrame(rows)
    out_path = out_dir / f"cn_vs_global_pairwise_{sector}_2000_2013_new.parquet"
    df_sector.to_parquet(out_path, index=False)

    sector_stats.append({
        "sector": sector,
        "n_cn": len(cn_grp),
        "n_global": len(gl_grp),
        "n_pairs_written": len(df_sector),
        "out_file": out_path.name,
    })

df_sector_stats = pd.DataFrame(sector_stats)
stats_path = out_dir / "sector_pair_counts_new.csv"
df_sector_stats.to_csv(stats_path, index=False)

print("Wrote per-sector parquet files to:", out_dir)
print("Wrote sector counts to:", stats_path)
display(df_sector_stats.sort_values("n_pairs_written", ascending=False).head(10))

PATH_TO_LOCAL FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
W0105 14:04:59.363000 92782 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Common sectors: 167


Sectors:   0%|          | 0/167 [00:00<?, ?it/s]

Wrote per-sector parquet files to: PATH_TO_ENTTEMPLATES_DATA_ROOT/Analysis/python_Patent/output/cn_vs_global_pairwise_similarity_2000_2013_by_sector_new
Wrote sector counts to: PATH_TO_ENTTEMPLATES_DATA_ROOT/Analysis/python_Patent/output/cn_vs_global_pairwise_similarity_2000_2013_by_sector_new/sector_pair_counts_new.csv


,sector,n_cn,n_global,n_pairs_written,out_file
142,MobilityTechOpticalProductOpticalProduct,2022,16054,32461188,cn_vs_global_pairwise_MobilityTechOpticalProdu...
11,AIMLVerticalApplicationsIndustrial,1066,14328,15273648,cn_vs_global_pairwise_AIMLVerticalApplications...
107,IoTConnectedBuildingsSmartHome,2141,6212,13299892,cn_vs_global_pairwise_IoTConnectedBuildingsSma...
124,MobilityTechCarDealershipOperatorCarDealership...,724,11807,8548268,cn_vs_global_pairwise_MobilityTechCarDealershi...
149,MobilityTechVideoRecordPlatformVideoRecordPlat...,952,6258,5957616,cn_vs_global_pairwise_MobilityTechVideoRecordP...
119,MobilityTechAircraftManagementAircraftManagement,644,7010,4514440,cn_vs_global_pairwise_MobilityTechAircraftMana...
99,InfoSecSecurityOperationsManagedSecurityServices,843,3892,3280956,cn_vs_global_pairwise_InfoSecSecurityOperation...
147,MobilityTechSolarPowerSolarPower,640,4296,2749440,cn_vs_global_pairwise_MobilityTechSolarPowerSo...
2,AIMLAIMLSemiconductorsProcessorDesign,548,4097,2245156,cn_vs_global_pairwise_AIMLAIMLSemiconductorsPr...
6,AIMLHorizontalPlatformsComputerVision,285,7402,2109570,cn_vs_global_pairwise_AIMLHorizontalPlatformsC...


In [2]:
# Compile all sectors for CN-global together with a single concat at the end
sector_dir = output_dir / "cn_vs_global_pairwise_similarity_2000_2013_by_sector_new"
df_list = []

for sector_file in sector_dir.glob("cn_vs_global_pairwise_*_2000_2013_new.parquet"):
    print(f"Loading sector file: {sector_file.name}")
    df_list.append(pd.read_parquet(sector_file))

df_all_sectors = pd.concat(df_list, ignore_index=True)

Loading sector file: cn_vs_global_pairwise_AIMLHorizontalPlatformsNaturalLanguageTechnology_2000_2013_new.parquet
Loading sector file: cn_vs_global_pairwise_IoTConnectedBuildingsSmartHome_2000_2013_new.parquet
Loading sector file: cn_vs_global_pairwise_FoodTechIndustrialconsumertechAdvancedvending_2000_2013_new.parquet
Loading sector file: cn_vs_global_pairwise_FoodTechDiscoveryreviewFoodbeveragediscovery_2000_2013_new.parquet
Loading sector file: cn_vs_global_pairwise_InsurtechPropertyandCasualtyPetinsurance_2000_2013_new.parquet
Loading sector file: cn_vs_global_pairwise_InfoSecIdentityAccessManagementAccessManagement_2000_2013_new.parquet
Loading sector file: cn_vs_global_pairwise_EdTechEarlyEducationSolutionsforParents_2000_2013_new.parquet
Loading sector file: cn_vs_global_pairwise_AIMLVerticalApplicationsFinancialServices_2000_2013_new.parquet
Loading sector file: cn_vs_global_pairwise_MobilityTechAutonomousDrivingSoftwareTeleoperation_2000_2013_new.parquet
Loading sector file: c

In [5]:
df_all_sectors["fullname_raw"] = df_all_sectors["sector"].str.lower()
# rename disambig_country to country_2digit
df_all_sectors = df_all_sectors.rename(columns={"disambig_country": "country_2digit"})
regression_data = pd.read_stata("PATH_TO_ENTTEMPLATES_DATA_ROOT/Analysis/Stata/data_v2/regression_corrected_120623.dta")
regression_data = regression_data[["fullname_raw", "subsegment", "subsegment1", "hqcountry", "suitability_score_wdi", "OECD_b80s","country_2digit"]].drop_duplicates()

df_all_sectors_with_macrosector = pd.merge(
    df_all_sectors,
    regression_data,
    on=["fullname_raw","country_2digit"],
    how="inner",
)

/var/folders/p9/3r6trd5n5t7127t0ljj8z63h0000gn/T/ipykernel_97786/3214929686.py:4: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  regression_data = pd.read_stata("PATH_TO_ENTTEMPLATES_DATA_ROOT/Analysis/Stata/data_v2/regression_corrected_120623.dta")


In [8]:
agg_df = (
    df_all_sectors_with_macrosector.groupby(["hqcountry", "subsegment", "subsegment1" ,"suitability_score_wdi"])["sbert_score"]
    .agg(
        mean="mean",
        median="median",
        top25=lambda x: x.quantile(0.75),
        top10=lambda x: x.quantile(0.90),
        top5=lambda x: x.quantile(0.95),
        top1=lambda x: x.quantile(0.99),
        max="max",
    )
    .reset_index()
)

agg_df.head()

,hqcountry,subsegment,subsegment1,suitability_score_wdi,mean,median,top25,top10,top5,top1,max
0,Argentina,AI ML|Vertical Applications|Healthcare,AI ML,0.871483,0.153847,0.154913,0.206250,0.249670,0.278849,0.331725,0.368662
1,Argentina,AgTech|Ag biotech|Biomaterials,AgTech,1.173765,0.096275,0.085342,0.146244,0.208000,0.227878,0.291861,0.336086
2,Argentina,AgTech|Ag biotech|Plant data & analysis,AgTech,1.173765,0.167635,0.171542,0.218160,0.250402,0.275279,0.341928,0.376786
3,Argentina,AgTech|Precision ag|Field IoT,AgTech,1.173765,0.130839,0.125242,0.210774,0.252793,0.286736,0.329688,0.409846
4,Argentina,Carbon and Emissions Tech|Carbon Tech|Point So...,Carbon and Emissions Tech,0.916271,0.098487,0.082537,0.148852,0.206238,0.243832,0.353874,0.409846


In [ ]:
agg_df.to_stata(output_dir/'patent_cn_global_similarity_by_country_sector_new.dta', write_index=False)